In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tfmplayground.mouse.evaluation import (
    evaluate,
    get_model,
    plot_metric,
    plot_prediction,
    predict_mouse,
)
from tfmplayground.mouse.features import (
    Feature,
    add_time_features,
    list_features,
    model_columns,
    to_model_input,
    to_timeseries,
)
from tfmplayground.mouse.preprocessing import (
    THERAPY_TYPES,
    PreprocessingConfig,
    build_info_df,
    split_context_test,
)
from tfmplayground.mouse import michael, preprocessing

logging.basicConfig(level=logging.INFO, format="%(message)s")

In [ ]:
# DATAPATH = preprocessing.RAW_PICKLE_PATH  # "/data/mice/maus.pandas.pkl"
# load_dataset = preprocessing.load_dataset
# config = PreprocessingConfig(
#     trails=[3],
#     roles=[1, 2, 3],
#     min_length=5,
#     max_weight=100,
#     num_animals=4000,
#     num_test_animals=4000,
#     prediction_length=2,
#     include_surgery=False,
#     include_scores=False,
#     seed=None,
# )

DATAPATH = preprocessing.RAW_PICKLE_PATH  # "/data/mice/maus.pandas.pkl"
load_dataset = michael.load_dataset
config = PreprocessingConfig(
    trails=[3],
    roles=[1, 2, 3],
    min_length=5,
    max_weight=100,
    num_animals=4000,
    num_test_animals=4000,
    prediction_length=1,
    seed=None,
)

In [ ]:
PREDICTION_LENGTH = config.prediction_length

# Time features, see list_features() for all options
FEATURES = [
    Feature.RUNNING_INDEX,
    Feature.DAYS_SINCE,
    Feature.SEASONAL
]
X_COLUMNS = model_columns(FEATURES)

In [ ]:
# Results tables, every model section appends its rows
RESULT_COLUMNS = ["Model", *THERAPY_TYPES.values(), "Overall"]
table_mae = pd.DataFrame(columns=RESULT_COLUMNS)
table_mape = pd.DataFrame(columns=RESULT_COLUMNS)

## Prepare data

### Read data

In [ ]:
df = load_dataset(DATAPATH, config)
info_df = build_info_df(df)

df

### Preprocess data

In [ ]:
context_df, future_df, test_df = split_context_test(df, config)

context_df

In [ ]:
context_tsdf, future_tsdf = to_timeseries(context_df, future_df)
train_tsdf, test_tsdf = add_time_features(context_tsdf, future_tsdf, FEATURES)

# Input of the self-trained PFNs
pfn_train_df = to_model_input(train_tsdf, FEATURES)
pfn_test_df = to_model_input(test_tsdf, FEATURES)

## Analysis

### Basic mouse
PFN trained on only the mouse prior.

In [ ]:
MODELPATH = "/data/PFN/Mouse/Experiments/Mouse1/"
MODELFILE = "pretrained_mousepfn.pth"
MODELNAME = "MousePFN"
BUCKETNAME = "buckets_mousepfn.pth"

reg = get_model(
    model_path=MODELPATH,
    model_file=MODELFILE,
    bucket_name=BUCKETNAME
)

table_mae, table_mape = evaluate(
    model_name=MODELNAME,
    reg=reg,
    table_mae=table_mae,
    table_mape=table_mape,
    info_df=info_df,
    train_tsdf=pfn_train_df,
    test_tsdf=pfn_test_df,
    test_df=test_df,
    x_columns=X_COLUMNS,
    prediction_length=PREDICTION_LENGTH,
    auto_regressive=False,
    length_boundary=None
)

In [ ]:
NUM_PLOTS = 1

mouse_ids = pfn_train_df["id"].unique()
plot_mouse_ids = np.random.choice(mouse_ids, size=NUM_PLOTS, replace=False)

for id in plot_mouse_ids:
    train_x, train_y, test_x, test_y, preds = predict_mouse(
        id,
        reg,
        info_df,
        pfn_train_df,
        pfn_test_df,
        test_df,
        x_columns=X_COLUMNS,
        prediction_length=PREDICTION_LENGTH,
        auto_regressive=False,
        length_boundary=None,
        for_plotting=True
    )

    plot_prediction(
        train_x,
        train_y,
        test_x,
        test_y,
        preds,
        x_columns=X_COLUMNS,
    )

### TabPFN -> mouse
Pretrained on TabPFN prior v1 then pretrained on mouse prior.

In [ ]:
MODELPATH = "/data/PFN/Mouse/Experiments/TabPFN_Mouse1/"
MODELFILE = "pretrained_mousepfn.pth"
MODELNAME = "TabPFN-MousePFN"
BUCKETNAME = "buckets_mousepfn.pth"

reg = get_model(
    model_path=MODELPATH,
    model_file=MODELFILE,
    bucket_name=BUCKETNAME
)

table_mae, table_mape = evaluate(
    model_name=MODELNAME,
    reg=reg,
    table_mae=table_mae,
    table_mape=table_mape,
    info_df=info_df,
    train_tsdf=pfn_train_df,
    test_tsdf=pfn_test_df,
    test_df=test_df,
    x_columns=X_COLUMNS,
    prediction_length=PREDICTION_LENGTH,
    auto_regressive=False,
    length_boundary=None
)

In [ ]:
NUM_PLOTS = 1

mouse_ids = pfn_train_df["id"].unique()
plot_mouse_ids = np.random.choice(mouse_ids, size=NUM_PLOTS, replace=False)

for id in plot_mouse_ids:
    train_x, train_y, test_x, test_y, preds = predict_mouse(
        id,
        reg,
        info_df,
        pfn_train_df,
        pfn_test_df,
        test_df,
        x_columns=X_COLUMNS,
        prediction_length=PREDICTION_LENGTH,
        auto_regressive=False,
        length_boundary=None,
        for_plotting=True
    )

    plot_prediction(
        train_x,
        train_y,
        test_x,
        test_y,
        preds,
        x_columns=X_COLUMNS,
    )

### Mouse, no context
Make the context window only contain the mouse we're predicting for. Do not add other similar mice.

In [ ]:
MODELPATH = "/data/PFN/Mouse/Experiments/Mouse1/"
MODELFILE = "pretrained_mousepfn.pth"
MODELNAME = "Auto-reg MousePFN"
BUCKETNAME = "buckets_mousepfn.pth"

reg = get_model(
    model_path=MODELPATH,
    model_file=MODELFILE,
    bucket_name=BUCKETNAME
)

table_mae, table_mape = evaluate(
    model_name=MODELNAME,
    reg=reg,
    table_mae=table_mae,
    table_mape=table_mape,
    info_df=info_df,
    train_tsdf=pfn_train_df,
    test_tsdf=pfn_test_df,
    test_df=test_df,
    x_columns=X_COLUMNS,
    prediction_length=PREDICTION_LENGTH,
    auto_regressive=True,
    length_boundary=None
)

In [ ]:
NUM_PLOTS = 1

mouse_ids = pfn_train_df["id"].unique()
plot_mouse_ids = np.random.choice(mouse_ids, size=NUM_PLOTS, replace=False)

for id in plot_mouse_ids:
    train_x, train_y, test_x, test_y, preds = predict_mouse(
        id,
        reg,
        info_df,
        pfn_train_df,
        pfn_test_df,
        test_df,
        x_columns=X_COLUMNS,
        prediction_length=PREDICTION_LENGTH,
        auto_regressive=True,
        length_boundary=None,
        for_plotting=True
    )

    plot_prediction(
        train_x,
        train_y,
        test_x,
        test_y,
        preds,
        x_columns=X_COLUMNS,
    )

### Mouse, context for long-lived mice
Only add context when the mice has a lot of data, otherwise do it auto-regressively.

In [ ]:
LENGTH_BOUNDARY = 20
MODELPATH = "/data/PFN/Mouse/Experiments/Mouse1/"
MODELFILE = "pretrained_mousepfn.pth"
MODELNAME = f"<{LENGTH_BOUNDARY} Auto-reg MousePFN"
BUCKETNAME = "buckets_mousepfn.pth"

reg = get_model(
    model_path=MODELPATH,
    model_file=MODELFILE,
    bucket_name=BUCKETNAME
)

table_mae, table_mape = evaluate(
    model_name=MODELNAME,
    reg=reg,
    table_mae=table_mae,
    table_mape=table_mape,
    info_df=info_df,
    train_tsdf=pfn_train_df,
    test_tsdf=pfn_test_df,
    test_df=test_df,
    x_columns=X_COLUMNS,
    prediction_length=PREDICTION_LENGTH,
    auto_regressive=False,
    length_boundary=LENGTH_BOUNDARY
)

In [ ]:
NUM_PLOTS = 1

mouse_ids = pfn_train_df["id"].unique()
plot_mouse_ids = np.random.choice(mouse_ids, size=NUM_PLOTS, replace=False)

for id in plot_mouse_ids:
    train_x, train_y, test_x, test_y, preds = predict_mouse(
        id,
        reg,
        info_df,
        pfn_train_df,
        pfn_test_df,
        test_df,
        x_columns=X_COLUMNS,
        prediction_length=PREDICTION_LENGTH,
        auto_regressive=False,
        length_boundary=LENGTH_BOUNDARY,
        for_plotting=True
    )

    plot_prediction(
        train_x,
        train_y,
        test_x,
        test_y,
        preds,
        x_columns=X_COLUMNS,
    )

### Comparison

In [ ]:
table_mae.sort_values("Overall").reset_index()
table_mape.sort_values("Overall").reset_index()

# MAE plot
plot_metric(table_mae, "MAE")

# MAPE plot
plot_metric(table_mape, "MAPE")

plt.show()